In [1]:
"""
extract_curve_features.py

WHAT THIS DOES:
Our first attempt used simple summary numbers (capacity, resistance) and
they turned out to be weak predictors (see docs/05_evaluation.md).
This script builds the feature the original researchers found actually
works: the VARIANCE of the difference between the discharge dQ/dV curve
at cycle 100 and cycle 10.

WHY THIS SHOULD WORK BETTER (the physical intuition):
A single capacity number at cycle 100 tells you "how much energy this
cell holds right now." It doesn't tell you HOW the cell is degrading -
two cells could have the identical capacity at cycle 100 but be
degrading through completely different internal mechanisms (e.g. lithium
plating vs. electrode cracking), which affect long-term life differently.
The SHAPE of the discharge curve (specifically dQ/dV, how capacity
changes as voltage changes) encodes information about which internal
mechanism is happening - so how much that SHAPE has changed by cycle 100
is a much richer signal than a single capacity number.

HOW TO RUN:
Same as extract_summary.py - edit FILE_PATH, run, upload the resulting CSV.
"""

import h5py
import pandas as pd
import numpy as np

file_path = r"C:\Users\jeffe\OneDrive\Desktop\Portfolio Projects\archive\2018-04-12_batchdata_updated_struct_errorcorrect.mat"


def extract_curve_features(file_path):
    rows = []

    with h5py.File(file_path, 'r') as f:
        batch = f['batch']
        num_cells = batch['summary'].shape[0]
        print(f"Found {num_cells} battery cells in this file.")

        for i in range(num_cells):
            try:
                # --- cycle_life (same as before) ---
                cl_ref = f[batch['cycle_life'][i, 0]]
                cycle_life = int(np.array(cl_ref)[0][0])

                # --- navigate into this cell's cycles ---
                cycles_ref = f[batch['cycles'][i, 0]]
                dqdv_field = cycles_ref['discharge_dQdV']

                # Skip any cell that doesn't have at least 100 cycles recorded
                if dqdv_field.shape[0] < 100:
                    print(f"Skipped cell {i}: fewer than 100 cycles recorded")
                    continue

                # Follow the reference for cycle 10 (index 9) and cycle 100 (index 99)
                cycle10_curve = np.array(f[dqdv_field[9, 0]]).flatten()
                cycle100_curve = np.array(f[dqdv_field[99, 0]]).flatten()

                # The actual feature: variance of the point-by-point difference
                diff_curve = cycle100_curve - cycle10_curve
                dqdv_variance = np.var(diff_curve)

                rows.append({
                    "cell_id": i,
                    "cycle_life": cycle_life,
                    "dQdV_variance_10_100": dqdv_variance,
                })
            except Exception as e:
                print(f"Skipped cell {i}, had an issue: {e}")

    return pd.DataFrame(rows)


if __name__ == "__main__":
    df = extract_curve_features(file_path)
    print(df.head())
    print(f"\nExtracted {len(df)} cells.")
    df.to_csv("curve_features.csv", index=False)
    print("Saved to curve_features.csv - upload this one to Claude.")

Found 46 battery cells in this file.
Skipped cell 23, had an issue: cannot convert float NaN to integer
Skipped cell 32, had an issue: cannot convert float NaN to integer
   cell_id  cycle_life  dQdV_variance_10_100
0        0        1009              0.001857
1        1        1063              0.002698
2        2        1267              0.005658
3        3        1115              0.388658
4        4        1048              0.012072

Extracted 44 cells.
Saved to curve_features.csv - upload this one to Claude.


In [2]:
import numpy as np

df["log_dQdV_variance"] = np.log(df["dQdV_variance_10_100"])
log_corr = df["log_dQdV_variance"].corr(df["cycle_life"])
print(f"Correlation after log transform: {log_corr:.3f}")

Correlation after log transform: -0.049
